# 04 - Precipitação: GPM IMERG diário

| | |
|---|---|
| **Tarefa do plano** | Apoia a S1-06 e a verificação de sobreposição CHIRPS x IMERG (semana 2) |
| **Variáveis (Parte I)** | #27 precipitação GPM IMERG |
| **Fonte** | GPM IMERG V07 (NASA) via Earth Engine `NASA/GPM_L3/IMERG_V07` |
| **Saída** | `data/raw/precipitation/imerg/` |

A fonte é de meia em meia hora (mm/h). Cada dia é a taxa média do dia UTC vezes 24, o que dá mm/dia. Um NetCDF por mês, grelha nativa de 0,1 graus. Os dias são dias UTC, enquanto os dias do CHIRPS são dias locais dos pluviómetros: comparar os dois em somas de vários dias.

Correr primeiro `00_setup_and_authentication` e `01_study_area_aoi`. Todos os
downloads podem ser retomados: os ficheiros que já existem são saltados, e cada
ficheiro fica registado em `data/metadata/download_manifest.csv`.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config" / "config.yaml").exists())
sys.path.insert(0, str(ROOT / "src"))

import ee
import pandas as pd
import matplotlib.pyplot as plt

from ete import config, datasets, gee, series
from ete import grid as G

cfg = config.load_config()
gee.init(cfg)

aoi = G.load_aoi(cfg)                       # AOI das imagens (EPSG:4326)
region = gee.ee_geometry(aoi)
grid30 = G.load_grid(cfg)                   # grelha do projeto a 30 m (EPSG:32736)
grid10 = G.load_grid(cfg, 10)               # grelha a 10 m, encaixada na de 30 m
climate_bounds = tuple(G.load_aoi(cfg, "aoi_climate").total_bounds)   # O, S, E, N
MASK = gee.ee_geometry(G.imagery_mask(cfg))  # distrito + margem: fora daqui as imagens mensais ficam sem dados

## 1. Download

In [ ]:
ds = cfg["datasets"]["imerg"]
OUT_DIR = config.path(cfg, "imerg")
grid = gee.collection_grid(ee.ImageCollection(ds["id"]), ds["band"], climate_bounds, fallback_res=0.1)
print(grid)

log = gee.download_daily_netcdf(cfg, "imerg", ds["id"], ds["band"], grid, OUT_DIR,
                                start=ds["start"], freq="MS", multiply=24, units="mm/day")
log["status"].value_counts()

## 2. Controlo de qualidade

In [ ]:
rain = series.open_series(cfg, "imerg", ds["band"])
print("período:", str(rain.time.values[0])[:10], "a", str(rain.time.values[-1])[:10], "|", rain.sizes)
print("dias em falta:", series.missing_days(rain))

daily = series.area_mean(rain, G.load_aoi(cfg, "basin")).rename("basin_mm")
csv = config.path(cfg, "interim", "precipitation", "imerg_daily_area_mean.csv")
daily.to_csv(csv, index_label="date")
daily.resample("MS").sum().plot(figsize=(13, 3.5), title="IMERG - precipitação mensal média da bacia (mm)")